# 06 VQE With Local Statevector

Goal: train a small variational circuit with PyTorch autograd and FlagQuantum's local statevector path.

Use this tutorial when you want a compact VQE workflow that is easy to inspect. The matching runnable example is `examples/single_machine_quantum_ai/01_vqe_statevector.py`.

## Runtime note

This notebook uses a single-device local statevector simulation. It is not a distributed scalability example.

In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "flagquantum").exists():
        sys.path.insert(0, str(candidate))
        break

import torch
import flagquantum as fq

torch.manual_seed(7)

## Build a small Hamiltonian

The Hamiltonian below combines nearest-neighbor `ZZ` terms with transverse `X` fields. It is small enough to optimize quickly on CPU.

In [ ]:
n_qubits = 3
layers = 1

hamiltonian = fq.Hamiltonian(
    [
        *(fq.pauli_term(0.7, "ZZ", (i, i + 1)) for i in range(n_qubits - 1)),
        *(fq.pauli_term(-0.25, "X", (i,)) for i in range(n_qubits)),
    ]
)

## Define a hardware-efficient ansatz

A good tutorial ansatz is explicit: rotations first, entanglers second. For production examples, prefer the reusable scripts under `examples/single_machine_quantum_ai/`.

In [ ]:
def build_ansatz(parameters: torch.Tensor) -> fq.Circuit:
    circuit = fq.Circuit(n_qubits)
    cursor = 0
    for _ in range(layers):
        for wire in range(n_qubits):
            circuit.ry(wire, theta=parameters[cursor])
            cursor += 1
            circuit.rz(wire, theta=parameters[cursor])
            cursor += 1
        for wire in range(n_qubits - 1):
            circuit.cx(wire, wire + 1)
    return circuit

n_params = fq.hardware_efficient_parameter_count(n_qubits, layers)
parameters = (0.1 * torch.randn(n_params)).requires_grad_(True)
print(f"parameters: {n_params}")

## Train with PyTorch

The loss is the Hamiltonian expectation value of the current circuit.

In [ ]:
optimizer = torch.optim.Adam([parameters], lr=0.05)

for step in range(5):
    optimizer.zero_grad()
    loss = hamiltonian.expectation(build_ansatz(parameters)).sum()
    loss.backward()
    optimizer.step()
    print({"step": step + 1, "energy": float(loss.detach())})

## Inspect the runtime summary

The plan summary records that this is a single-device fast path. That makes the claim boundary explicit.

In [ ]:
trained_circuit = build_ansatz(parameters.detach())
trained_circuit.plan().summary()